# Week 9

In [1]:
import pandas as pd, numpy as np

In [3]:
def simulate(A=1, B=1, C=10, D=1000):
  W = np.random.normal(0,1,D)
  X = W+np.random.normal(0,B,D)
  Y = A*X-W+np.random.normal(0,C,D)
  return Y, X, W

In [ ]:
def simulate(A=1, B=1, C=10, D=1000):
    W = np.random.normal(0, 1, D)
    X = W + np.random.normal(0, B, D)
    Y = A * X - W + np.random.normal(0, C, D)
    return Y, X, W

def run_ols(Y, X_mat):
    # Add constant
    X_mat = np.column_stack((np.ones(len(Y)), X_mat))
    # beta = (X'X)^-1 X'Y
    xtx = X_mat.T @ X_mat
    xtx_inv = np.linalg.inv(xtx)
    beta = xtx_inv @ X_mat.T @ Y
    
    # residuals
    resid = Y - X_mat @ beta
    dof = len(Y) - X_mat.shape[1]
    sigma2 = np.sum(resid**2) / dof
    
    # SE
    se = np.sqrt(np.diag(sigma2 * xtx_inv))
    t_vals = beta / se
    return t_vals

def check_power(n_sims=10000):
    significant_count = 0
    for _ in range(n_sims):
        Y, X, W = simulate()
        t_vals = run_ols(Y, np.column_stack((X, W)))
        # t-value for X is at index 1
        if abs(t_vals[1]) > 1.96:
            significant_count += 1
    return significant_count / n_sims

if __name__ == "__main__":
    print(f"Power: {check_power()}")


Power: 0.8829


In [5]:
import numpy as np

def simulate(A=1, B=1, C=10, D=1000):
    W = np.random.normal(0, 1, D)
    X = W + np.random.normal(0, B, D)
    Y = A * X - W + np.random.normal(0, C, D)
    return Y, X, W

def run_ols_results(Y, X_mat):
    # Add constant
    X_mat = np.column_stack((np.ones(len(Y)), X_mat))
    # beta = (X'X)^-1 X'Y
    xtx = X_mat.T @ X_mat
    xtx_inv = np.linalg.inv(xtx)
    beta = xtx_inv @ X_mat.T @ Y
    
    resid = Y - X_mat @ beta
    dof = len(Y) - X_mat.shape[1]
    sigma2 = np.sum(resid**2) / dof
    se = np.sqrt(np.diag(sigma2 * xtx_inv))
    t_vals = beta / se
    return beta[1], t_vals[1]

def calculate_skew(data):
    n = len(data)
    mu = np.mean(data)
    sigma = np.std(data)
    m3 = np.sum((data - mu)**3) / n
    skew = m3 / (sigma**3)
    return skew

def run_simulation(n_sims=50000):
    coefs = []
    t_stats = []
    for _ in range(n_sims):
        Y, X, W = simulate()
        coef, t_stat = run_ols_results(Y, np.column_stack((X, W)))
        coefs.append(coef)
        t_stats.append(t_stat)
    
    coefs = np.array(coefs)
    t_stats = np.array(t_stats)
    
    skew_coef = calculate_skew(coefs)
    skew_t = calculate_skew(t_stats)
    return skew_coef, skew_t

if __name__ == "__main__":
    skew_coef, skew_t = run_simulation()
    print(f"Skew of coef estimate: {skew_coef}")
    print(f"Skew of t-statistic: {skew_t}")

if __name__ == "__main__":
    skew_val = run_simulation()
    print(f"Skew of estimate: {skew_val}")


Skew of coef estimate: 0.005015012631854735
Skew of t-statistic: 0.014405514682602961
Skew of estimate: (np.float64(-0.0017108447514935688), np.float64(0.004016926685616168))


# Week 10

In [25]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Read the data
df = pd.read_csv('homework_10.1.csv')

# Dictionary to store fixed effects for each time period
fixed_effects_time = {}

# Loop through each time period (0 through 11)
for t in range(12):
    # Filter data for current time period
    df_time = df[df['time'] == t]
    
    # Prepare X (independent variable) and y (dependent variable)
    X = df_time['X']
    y = df_time['y']
    
    # Add constant to X for the regression
    X = sm.add_constant(X)
    
    # Run OLS regression
    model = sm.OLS(y, X).fit()
    
    # Extract the fixed effect (constant term)
    fixed_effect = model.params['const']
    fixed_effects_time[t] = fixed_effect
    
    print(f"Time {t}: Fixed Effect = {fixed_effect:.4f}")

# Display all fixed effects
print("\nSummary of Fixed Effects by Time:")
for time, fe in fixed_effects_time.items():
    print(f"Time {time}: {fe:.4f}")

Time 0: Fixed Effect = 2.7500
Time 1: Fixed Effect = 4.7358
Time 2: Fixed Effect = 6.1291
Time 3: Fixed Effect = 7.1522
Time 4: Fixed Effect = 8.0738
Time 5: Fixed Effect = 9.0478
Time 6: Fixed Effect = 8.5079
Time 7: Fixed Effect = 8.4129
Time 8: Fixed Effect = 8.0655
Time 9: Fixed Effect = 7.4927
Time 10: Fixed Effect = 6.0798
Time 11: Fixed Effect = 4.7628

Summary of Fixed Effects by Time:
Time 0: 2.7500
Time 1: 4.7358
Time 2: 6.1291
Time 3: 7.1522
Time 4: 8.0738
Time 5: 9.0478
Time 6: 8.5079
Time 7: 8.4129
Time 8: 8.0655
Time 9: 7.4927
Time 10: 6.0798
Time 11: 4.7628


In [18]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Read the data
df = pd.read_csv('homework_10.1.csv')

# Dictionary to store fixed effects for each city
fixed_effects = {}

# Loop through each city (0 through 9)
for city in range(10):
    # Filter data for current city
    df_city = df[df['city'] == city]
    
    # Prepare X (independent variable) and y (dependent variable)
    X = df_city['X']
    y = df_city['y']
    
    # Add constant to X for the regression
    X = sm.add_constant(X)
    
    # Run OLS regression
    model = sm.OLS(y, X).fit()
    
    # Extract the fixed effect (constant term)
    fixed_effect = model.params['const']
    fixed_effects[city] = fixed_effect
    
    print(f"City {city}: Fixed Effect = {fixed_effect:.4f}")

# Display all fixed effects
print("\nSummary of Fixed Effects:")
for city, fe in fixed_effects.items():
    print(f"City {city}: {fe:.4f}")

City 0: Fixed Effect = 11.0177
City 1: Fixed Effect = 5.2059
City 2: Fixed Effect = 7.9223
City 3: Fixed Effect = 12.9354
City 4: Fixed Effect = 10.8371
City 5: Fixed Effect = -0.0788
City 6: Fixed Effect = 7.6922
City 7: Fixed Effect = 3.2779
City 8: Fixed Effect = 3.3835
City 9: Fixed Effect = 5.4626

Summary of Fixed Effects:
City 0: 11.0177
City 1: 5.2059
City 2: 7.9223
City 3: 12.9354
City 4: 10.8371
City 5: -0.0788
City 6: 7.6922
City 7: 3.2779
City 8: 3.3835
City 9: 5.4626


In [22]:
import numpy as np
import statsmodels.api as sm

# Run multiple simulations to get stable coefficient estimates
n_simulations = 100
coef_X_list = []
coef_Z_list = []

for sim in range(n_simulations):
    # Generate the data as specified in the problem
    num = 10000
    
    X = np.clip(np.random.normal(3, 1, (num,)), 0.01, 10000)
    Z = np.clip(np.random.normal(3, 1, (num,)), 0.01, 10000)
    Y = np.log(X + Z) + np.random.normal(0, 1, (num,))
    
    # Model np.exp(Y) as a function of X and Z
    exp_Y = np.exp(Y)
    
    # Create design matrix with X and Z
    X_matrix = np.column_stack([X, Z])
    X_matrix = sm.add_constant(X_matrix)
    
    # Run OLS regression
    model = sm.OLS(exp_Y, X_matrix).fit()
    
    coef_X_list.append(model.params[1])
    coef_Z_list.append(model.params[2])

# Calculate average coefficients
avg_coef_X = np.mean(coef_X_list)
avg_coef_Z = np.mean(coef_Z_list)

print(f"Average coefficient of X over {n_simulations} simulations: {avg_coef_X:.4f}")
print(f"Average coefficient of Z over {n_simulations} simulations: {avg_coef_Z:.4f}")
print(f"\nAverage of both coefficients: {(avg_coef_X + avg_coef_Z)/2:.4f}")

Average coefficient of X over 100 simulations: 1.6583
Average coefficient of Z over 100 simulations: 1.6436

Average of both coefficients: 1.6509


Method 1: Standard error from OLS regression
Standard error of X's coefficient: 0.0351
X coefficient: 1.5247

Method 2: Standard deviation from 100 simulations
Standard deviation of X coefficient estimates: 0.0753
Mean of X coefficient estimates: 1.4919

Comparison:
Method 1 (OLS standard error): 0.0351
Method 2 (Simulation std dev): 0.0753
Difference: 0.0402

Answer: C. 2) is significantly bigger than 1), but neither is zero.
